# Image Data Processing

This notebook loads images from 10 folders and normalizes pixel values to the range [0, 1] by dividing by 255.0.

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
# Define the base directory containing 10 sub-folders of images
base_dir = 'data'  # Update this path to your dataset root

# Supported image extensions
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif')

In [ ]:
def load_and_normalize_images(folder_path):
    """
    Load all images from a folder and normalize pixel values to [0, 1].
    Handles 8-bit (uint8), 16-bit (uint16), and float images.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing images.

    Returns
    -------
    list of numpy.ndarray
        List of normalized image arrays with dtype float64.
    """
    if not os.path.isdir(folder_path):
        print(f'Warning: "{folder_path}" is not a valid directory. Skipping.')
        return []
    images = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.lower().endswith(IMAGE_EXTENSIONS):
            img_path = os.path.join(folder_path, filename)
            try:
                with Image.open(img_path) as img:
                    image = np.array(img)
                # Normalize to [0, 1] based on image dtype
                if image.dtype == np.uint8:
                    image = image / 255.0
                elif image.dtype == np.uint16:
                    image = image / 65535.0
                elif np.issubdtype(image.dtype, np.floating):
                    image = np.clip(image, 0.0, 1.0)
                else:
                    image = image / 255.0
                images.append(image)
            except Exception as e:
                print(f'Warning: could not load "{img_path}": {e}')
    return images

In [ ]:
# Load and normalize images from all 10 folders
all_images = {}  # dict mapping folder name -> list of normalized images

if not os.path.isdir(base_dir):
    print(f'base_dir "{base_dir}" does not exist. Please update the path.')
else:
    folders = sorted(
        [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]
    )[:10]  # take up to 10 folders

    print(f'Found {len(folders)} folder(s): {folders}')

    for folder in folders:
        folder_path = os.path.join(base_dir, folder)
        normalized = load_and_normalize_images(folder_path)
        all_images[folder] = normalized
        print(f'  Folder "{folder}": loaded {len(normalized)} image(s)')

In [ ]:
# Display one sample image per folder (first image, if available)
n_folders = len(all_images)
if n_folders == 0:
    print('No folders found or base_dir not set correctly.')
else:
    cols = min(5, n_folders)
    rows = (n_folders + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).flatten()
    for ax, (folder, images) in zip(axes, all_images.items()):
        if images:
            ax.imshow(images[0], cmap='gray' if images[0].ndim == 2 else None)
            ax.set_title(f'{folder}\n(normalized)', fontsize=9)
        else:
            ax.set_title(f'{folder}\n(no images)', fontsize=9)
        ax.axis('off')
    # Hide any unused axes
    for ax in axes[n_folders:]:
        ax.set_visible(False)
    plt.suptitle('Sample Normalized Images (pixel values in [0, 1])', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Verify normalization: pixel values should be in [0, 1]
for folder, images in all_images.items():
    for img in images:
        assert img.min() >= 0.0, f'Min value below 0 in folder {folder}'
        assert img.max() <= 1.0, f'Max value above 1 in folder {folder}'

print('All images are correctly normalized to [0, 1].')